# Бонусное задание. PySpark

Ноутбук запускается внутри контейнера `jupyter/pyspark-notebook`.
Датасеты доступны по пути `/home/jovyan/datasets/`.

In [1]:
import sys
sys.path.insert(0, "/usr/local/spark/python")
sys.path.insert(0, "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip")

## Задание 11. Загрузка данных и инспекция схемы

In [2]:
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("OlistAnalysis").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

DATASETS = "/home/jovyan/datasets"

In [3]:
customers_raw = spark.read.csv(f"{DATASETS}/olist_customers_dataset.csv", header=True)
orders_raw    = spark.read.csv(f"{DATASETS}/olist_orders_dataset.csv",    header=True)
payments_raw  = spark.read.csv(f"{DATASETS}/olist_order_payments_dataset.csv", header=True)

customers_raw.printSchema()
orders_raw.printSchema()
payments_raw.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: string (nullable = true)
 |-- payment_value: string (nullable = true)



In [4]:
customers_raw.show(3)
orders_raw.show(3)
payments_raw.show(3)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                   09790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                   01151|           sao paulo|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 3 rows

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_

In [5]:
orders_df = (
    orders_raw
    .withColumn("order_purchase_timestamp",      F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at",             F.to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date",  F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))
)

orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [6]:
print(f"Customers до очистки:  {customers_raw.count()}")
customers_df = customers_raw.na.drop()
print(f"Customers после очистки: {customers_df.count()}")

print(f"Payments до очистки:   {payments_raw.count()}")
payments_df = payments_raw.na.drop()
print(f"Payments после очистки:  {payments_df.count()}")

Customers до очистки:  99441
Customers после очистки: 99441
Payments до очистки:   103886
Payments после очистки:  103886


## Задание 12. Группировки и ранжирование

### Бизнес-задача 1. Заказы по год-месяц и статусу

In [7]:
orders_with_ym = orders_df.withColumn(
    "order_year_month",
    F.date_format(F.col("order_purchase_timestamp"), "y-M"),
)

answer_1 = (
    orders_with_ym
    .groupBy("order_year_month", "order_status")
    .count()
    .withColumnRenamed("count", "No_of_orders_year_month")
    .orderBy("order_year_month")
)

answer_1.show(10)

+----------------+------------+-----------------------+
|order_year_month|order_status|No_of_orders_year_month|
+----------------+------------+-----------------------+
|         2016-10|    canceled|                     24|
|         2016-10|    invoiced|                     18|
|         2016-10|  processing|                      2|
|         2016-10|     shipped|                      8|
|         2016-10|   delivered|                    265|
|         2016-10| unavailable|                      7|
|         2016-12|   delivered|                      1|
|          2016-9|     shipped|                      1|
|          2016-9|    canceled|                      2|
|          2016-9|   delivered|                      1|
+----------------+------------+-----------------------+
only showing top 10 rows



### Бизнес-задача 2. Клиенты по штатам с рангом

In [8]:
answer_2 = (
    customers_df
    .groupBy("customer_state")
    .count()
    .withColumnRenamed("count", "No_of_customers_state")
    .orderBy(F.col("No_of_customers_state").desc())
    .withColumn("rank", F.monotonically_increasing_id() + 1)
)

answer_2.show(27)

+--------------+---------------------+----+
|customer_state|No_of_customers_state|rank|
+--------------+---------------------+----+
|            SP|                41746|   1|
|            RJ|                12852|   2|
|            MG|                11635|   3|
|            RS|                 5466|   4|
|            PR|                 5045|   5|
|            SC|                 3637|   6|
|            BA|                 3380|   7|
|            DF|                 2140|   8|
|            ES|                 2033|   9|
|            GO|                 2020|  10|
|            PE|                 1652|  11|
|            CE|                 1336|  12|
|            PA|                  975|  13|
|            MT|                  907|  14|
|            MA|                  747|  15|
|            MS|                  715|  16|
|            PB|                  536|  17|
|            PI|                  495|  18|
|            RN|                  485|  19|
|            AL|                

## Задание 13. Оконные функции – повторные покупатели

### Бизнес-задача 3. Клиенты с 3+ заказами

In [9]:
orders_customers = orders_df.join(customers_df, on="customer_id", how="left")

repeat_buyers = (
    orders_customers
    .groupBy("customer_unique_id")
    .count()
    .where(F.col("count") >= 3)
)

print(f"Клиентов с 3+ заказами: {repeat_buyers.count()}")

Клиентов с 3+ заказами: 252


### Бизнес-задача 4. Дней между 1-м и 3-м заказом

In [10]:
repeat_ids = repeat_buyers.select("customer_unique_id")

w = Window.partitionBy("customer_unique_id").orderBy("order_purchase_timestamp")
ranked = (
    orders_customers
    .join(repeat_ids, on="customer_unique_id", how="inner")
    .withColumn("order_rank", F.row_number().over(w))
)

first_orders = ranked.where(F.col("order_rank") == 1).select(
    "customer_unique_id",
    F.col("order_purchase_timestamp").alias("first_order_ts"),
)
third_orders = ranked.where(F.col("order_rank") == 3).select(
    "customer_unique_id",
    F.col("order_purchase_timestamp").alias("third_order_ts"),
)

result = (
    first_orders
    .join(third_orders, on="customer_unique_id")
    .withColumn("days_to_third", F.datediff("third_order_ts", "first_order_ts"))
)

result.agg(
    F.avg("days_to_third").alias("avg_days"),
    F.min("days_to_third").alias("min_days"),
    F.max("days_to_third").alias("max_days"),
).show()

+------------------+--------+--------+
|          avg_days|min_days|max_days|
+------------------+--------+--------+
|131.54761904761904|       0|     633|
+------------------+--------+--------+

